# 5. Processors and backends

A **DSL processor** turns DSL operators into other callables; a **backend** is a processor returning its own DSL operators, which compute the same values in a different way. The definition is the same for all the backends.

```
FieldOperator (embedded) --backend(operator)--> FieldOperator of the backend --call--> Field
                         (dispatches on the kind: backend.field_operator / backend.scan_operator)
```

Contents: DSL processors · the backends · lazy values and backward propagation · backends on a mesh · writing a backend. Compilers are in "6. Compilation", allocators in "7. Allocation".

## DSL processors

| Concept | Extends | `field_operator(op)` returns | `scan_operator(op)` returns |
|---|---|---|---|
| `DSLProcessor` | | any callable | any callable |
| `DSLInterpreter` | `DSLProcessor` | `FieldOperator[P, R]` | `ScanOperator` |
| `DSLCompiler` | `DSLInterpreter` | `CompiledFieldOperator[P, R]` | `CompiledScanOperator` |
| `Backend` | `DSLInterpreter`, `AllocatorProvider` | `FieldOperator[P, R]` | `ScanOperator` |

Calling a processor **dispatches** on the kind of the operator: `processor(op)` calls `processor.field_operator(op)` or `processor.scan_operator(op)`, like the function `dispatch(processor, op)`. `__call__` is overloaded, so type checkers infer e.g. `eager(fop)` to be a `FieldOperator` with the signature of `fop`. An interpreter's operators may add keyword arguments with default values, e.g. `out=` or `domain=`: they still match the signature of the definition. The protocols implement `__call__`: classes subclassing them explicitly inherit it.

At runtime, `isinstance` only checks member names: it cannot tell a `DSLProcessor` from a `DSLInterpreter`, whose difference is in the return types, checked by type checkers only.

In [1]:
import numpy as np
from concepts import (
    Axis,
    Backend,
    Dims,
    DSLCompiler,
    DSLInterpreter,
    DSLProcessor,
    Field,
    dispatch,
    getsource,
)
from implementation import NumPyArrayField, field_operator, scan_operator


class I(Axis): ...


class K(Axis): ...


@field_operator()
def smooth(f: Field[Dims[I], float]) -> Field[Dims[I], float]:
    return 0.25 * f(I - 1) + 0.5 * f + 0.25 * f(I + 1)


@scan_operator(axis=K)
def total(carry: float, x: float) -> float:
    return carry + x


class LineCounter(DSLProcessor):
    """A `DSLProcessor` returning, for each operator, a function counting its source lines."""

    def field_operator(self, operator):
        return lambda: f"field operator, {len(getsource(operator).splitlines())} lines"

    def scan_operator(self, operator):
        return lambda: f"scan operator, {len(getsource(operator).splitlines())} lines"


counter = LineCounter()
print(counter(smooth)(), "|", dispatch(counter, total)())
print(
    "DSLProcessor, DSLInterpreter, DSLCompiler, Backend:",
    [isinstance(counter, p) for p in (DSLProcessor, DSLInterpreter, DSLCompiler, Backend)],
)

field operator, 3 lines | scan operator, 3 lines
DSLProcessor, DSLInterpreter, DSLCompiler, Backend: [True, True, True, False]


## The backends

The backends of `implementation.py` are all `Backend`s. They differ in how they compute the domain of the result, and when they compute its values (see "Semantics of domains, values and fields"):

| Backend | Domain computation | Values computed | Result |
|---|---|---|---|
| `eager` | forward | when called | `Field` |
| `lazy` | forward, then restricted to the requested domain | when first used | `DeferredField` |
| `lazy_backward` | backward, from the requested domain | when first used | `DeferredField` |
| `column_major` | forward | when called | `Field`, laid out column-major ("7. Allocation") |
| `compiled` | forward | when called | `Field`, from generated code ("6. Compilation") |

On every backend, a requested domain is checked against the result's domain: outside it, a `ValueError`. The operators print what they compute. All of them compute scans eagerly, level by level (see "4. Scan operators"). Simplified: the lazy backends only support field operators returning a single field, not tuples. `backend.__gt_allocator__` allocates fields for them, e.g. to write their results into.

In [2]:
from implementation import column_major, compiled, eager, lazy, lazy_backward

backends = (eager, lazy, lazy_backward, column_major, compiled)
signal = NumPyArrayField({I: range(0, 8)}, [0.0, 0.0, 4.0, 8.0, 4.0, 0.0, 0.0, 0.0])
for backend in backends:
    print(f"--- {backend} ---")
    op = backend(smooth)  # dispatches to `backend.field_operator(smooth)`
    assert op.__gt_definition__ is smooth.__gt_definition__
    result = op(signal)  # computed now, or only when first used
    print("is a Backend:", isinstance(backend, Backend), "| result:", type(result).__name__)
    out = backend.__gt_allocator__.zeros(signal.domain)
    out[{I: range(2, 5)}] = result  # only these values are read
    print("out =", out)
print("numpy: Backend", isinstance(np, Backend))  # an allocator, but no `__gt_allocator__`

--- eager ---
[eager] computed Field(I[1:7]: [1.0, 4.0, 6.0, 4.0, 1.0, 0.0])
is a Backend: True | result: NumPyArrayField
out = Field(I[0:8]: [0.0, 0.0, 4.0, 6.0, 4.0, 0.0, 0.0, 0.0])
--- lazy ---
is a Backend: True | result: DeferredField
[lazy] computed I[1:7], kept I[2:5]
out = Field(I[0:8]: [0.0, 0.0, 4.0, 6.0, 4.0, 0.0, 0.0, 0.0])
--- lazy_backward ---
is a Backend: True | result: DeferredField
[lazy_backward] computed I[2:5]
out = Field(I[0:8]: [0.0, 0.0, 4.0, 6.0, 4.0, 0.0, 0.0, 0.0])
--- column_major ---
[allocated] computed Field(I[1:7]: [1.0, 4.0, 6.0, 4.0, 1.0, 0.0])
is a Backend: True | result: NumPyArrayField
out = Field(I[0:8]: [0.0, 0.0, 4.0, 6.0, 4.0, 0.0, 0.0, 0.0])
--- compiled ---
[compiled] compiling smooth
[compiled] computed Field(I[1:7]: [1.0, 4.0, 6.0, 4.0, 1.0, 0.0])
is a Backend: True | result: NumPyArrayField
out = Field(I[0:8]: [0.0, 0.0, 4.0, 6.0, 4.0, 0.0, 0.0, 0.0])
numpy: Backend False


## Lazy values and backward propagation

A `DeferredField` has a domain, computed forward when the operator is called, but its values are only computed when first used, e.g. printed or read. Until then, `[...]` and requested domains narrow its domain at no cost.

When the input of an operator is itself a `DeferredField`, e.g. in a chain `op(op(x))`, `lazy_backward` propagates the requested domain backwards: each operator restricts its arguments to the requested domain grown by its extents, so the inner operator only computes what the outer one reads. `lazy` computes each operator on its whole forward domain and then keeps the requested part.

In [3]:
for backend in (lazy, lazy_backward):
    print(f"--- {backend} ---")
    op = backend(smooth)
    twice = op[{I: range(3, 5)}](op(signal))  # nothing computed yet
    print("domain before computing:", twice.domain[I])
    print(twice)
    try:
        backend(smooth)[{I: range(0, 3)}](signal)  # checked before computing
    except ValueError as error:
        print("ValueError:", error)

--- lazy ---
domain before computing: range(3, 5)
[lazy] computed I[1:7], kept I[1:7]
[lazy] computed I[2:6], kept I[3:5]
Field(I[3:5]: [5.0, 3.75])
ValueError: I[0:3] is not within the domain I[1:7].
--- lazy_backward ---
domain before computing: range(3, 5)
[lazy_backward] computed I[3:5]
[lazy_backward] computed I[2:6]
Field(I[3:5]: [5.0, 3.75])
ValueError: I[0:3] is not within the domain I[1:7].


## Backends on a mesh

All the backends work with relations and index sets (see "2. Connectivities"). The example: a ring of 6 cells, where edge `e` lies between cells `e` and `e + 1` (modulo 6). Two operators compute a gradient on the edges and a divergence on the cells, with *local fields* of signs; their chain is a discrete Laplacian.

- The lazy backends keep the mesh of the call: the values can be computed after the `with Mesh(...)` block.
- On a rank of a distributed run, the operators run the same way, in its `with local_mesh:` block: see "10. Local programs".
- `lazy_backward` propagates a requested set of cells back through the relations: only the edges and cells needed are computed, as index sets.
- The lazy backends compute domains from the extents, which follow the reads through local variables and nested operator calls (see "3. DSL definitions and operators"): the chain can also be written as one operator, below, and works on all the backends, with requested domains.

In [4]:
from concepts import Location, Mesh, Neighbor, Relation
from implementation import neighbor_table, sum_over


class Cell(Location): ...


class Edge(Location): ...


class E2C(Relation[Edge, Cell], max_neighbors=2): ...  # [left cell, right cell]


class C2E(Relation[Cell, Edge], max_neighbors=2): ...  # [left edge, right edge]


ring = {
    E2C: neighbor_table(E2C, [[e, (e + 1) % 6] for e in range(6)]),
    C2E: neighbor_table(C2E, [[(c - 1) % 6, c] for c in range(6)]),
}
right_minus_left = [[-1.0, 1.0]] * 6
edge_sign = NumPyArrayField({Edge: range(6), Neighbor[E2C]: range(2)}, right_minus_left)
cell_sign = NumPyArrayField({Cell: range(6), Neighbor[C2E]: range(2)}, right_minus_left)


@field_operator()
def gradient(
    u: Field[Dims[Cell], float], sign: Field[Dims[Edge, Neighbor[E2C]], float]
) -> Field[Dims[Edge], float]:
    return sum_over(sign * u(E2C), axis=Neighbor[E2C])


@field_operator()
def divergence(
    g: Field[Dims[Edge], float], sign: Field[Dims[Cell, Neighbor[C2E]], float]
) -> Field[Dims[Cell], float]:
    return sum_over(sign * g(C2E), axis=Neighbor[C2E])


u = NumPyArrayField({Cell: range(6)}, [0.0, 1.0, 4.0, 9.0, 16.0, 25.0])
results = {}
with Mesh(ring):
    for backend in backends:
        grad, div = backend(gradient), backend(divergence)
        results[backend] = div[{Cell: (2, 3)}](grad(u, edge_sign), cell_sign)
for backend, result in results.items():  # the lazy backends compute their values now
    print(f"--- {backend}: {result}")

[eager] computed Field(Edge[0:6]: [1.0, 3.0, 5.0, 7.0, 9.0, -25.0])
[eager] computed Field(Cell[0:6]: [26.0, 2.0, 2.0, 2.0, 2.0, -34.0])
[allocated] computed Field(Edge[0:6]: [1.0, 3.0, 5.0, 7.0, 9.0, -25.0])
[allocated] computed Field(Cell[0:6]: [26.0, 2.0, 2.0, 2.0, 2.0, -34.0])
[compiled] compiling gradient
[compiled] computed Field(Edge[0:6]: [1.0, 3.0, 5.0, 7.0, 9.0, -25.0])
[compiled] compiling divergence
[compiled] computed Field(Cell[0:6]: [26.0, 2.0, 2.0, 2.0, 2.0, -34.0])
--- eager: Field(Cell{2, 3}: [2.0, 2.0])
[lazy] computed Edge[0:6], kept Edge[0:6]
[lazy] computed Cell[0:6], kept Cell{2, 3}
--- lazy: Field(Cell{2, 3}: [2.0, 2.0])
[lazy_backward] computed Cell{2, 3}
[lazy_backward] computed Edge[1:4]
--- lazy_backward: Field(Cell{2, 3}: [2.0, 2.0])
--- column_major: Field(Cell{2, 3}: [2.0, 2.0])
--- compiled: Field(Cell{2, 3}: [2.0, 2.0])


In [5]:
@field_operator()
def laplacian(
    u: Field[Dims[Cell], float],
    es: Field[Dims[Edge, Neighbor[E2C]], float],
    cs: Field[Dims[Cell, Neighbor[C2E]], float],
) -> Field[Dims[Cell], float]:
    return divergence(gradient(u, es), cs)  # nested operator calls


@field_operator()
def second_neighbors(u: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    on_edges = sum_over(u(E2C), axis=Neighbor[E2C])  # a local variable on the edges
    return sum_over(on_edges(C2E), axis=Neighbor[C2E])  # through two relations


from implementation import backward_domain, format_domain

with Mesh(ring):
    print("compiled:", compiled(laplacian)(u, edge_sign, cell_sign))
    print("lazy:", str(lazy(laplacian)(u, edge_sign, cell_sign)))  # no requested domain
    # `u` is read through E2C, then C2E: Cell{2, 3} needs the edges 1-3, so the cells 1-4
    needed = backward_domain({Cell: (2, 3)}, laplacian.__gt_extents__["u"])
    print("u needed for Cell{2, 3}:", format_domain(needed))
    print("lazy_backward:", str(lazy_backward(laplacian)[{Cell: (2, 3)}](u, edge_sign, cell_sign)))
    print("eager:", eager(second_neighbors)(u))
    print("lazy:", str(lazy(second_neighbors)(u)))
    print("lazy_backward:", str(lazy_backward(second_neighbors)[{Cell: (2, 3)}](u)))

[compiled] compiling laplacian
[compiled] computed Field(Cell[0:6]: [26.0, 2.0, 2.0, 2.0, 2.0, -34.0])
compiled: Field(Cell[0:6]: [26.0, 2.0, 2.0, 2.0, 2.0, -34.0])
[lazy] computed Cell[0:6], kept Cell[0:6]
lazy: Field(Cell[0:6]: [26.0, 2.0, 2.0, 2.0, 2.0, -34.0])
u needed for Cell{2, 3}: Cell[1:5]
[lazy_backward] computed Cell{2, 3}
lazy_backward: Field(Cell{2, 3}: [2.0, 2.0])
[eager] computed Field(Cell[0:6]: [26.0, 6.0, 18.0, 38.0, 66.0, 66.0])
eager: Field(Cell[0:6]: [26.0, 6.0, 18.0, 38.0, 66.0, 66.0])
[lazy] computed Cell[0:6], kept Cell[0:6]
lazy: Field(Cell[0:6]: [26.0, 6.0, 18.0, 38.0, 66.0, 66.0])
[lazy_backward] computed Cell{2, 3}
lazy_backward: Field(Cell{2, 3}: [18.0, 38.0])


## Writing a backend

A backend is any class with `field_operator`, `scan_operator`, `__gt_allocator__` and a dispatching `__call__`, inherited by subclassing `Backend` explicitly. `BaseBackend` of `implementation.py` also provides the allocator and an eager `scan_operator`, so a new backend may only need `field_operator`, returning any `FieldOperator`: here, an eager operator that also checks for non-finite values.

In [6]:
from implementation import BaseBackend, EagerFieldOperator


class CheckedFieldOperator(EagerFieldOperator):
    def _compute(self, *args, **kwargs):
        result = super()._compute(*args, **kwargs)
        assert np.isfinite(result.data).all(), f"non-finite values in {self.__name__}"
        return result


class CheckedInterpreter(BaseBackend):
    name = "checked"

    def field_operator(self, operator):
        return CheckedFieldOperator(operator.__gt_definition__)


checked = CheckedInterpreter()
print("Backend:", isinstance(checked, Backend), "| scan:", type(checked(total)).__name__)
checked(smooth)(signal)
try:
    checked(smooth)(NumPyArrayField({I: range(0, 3)}, [1.0, np.inf, 1.0]))
except AssertionError as error:
    print("AssertionError:", error)

Backend: True | scan: EagerScanOperator
[eager] computed Field(I[1:7]: [1.0, 4.0, 6.0, 4.0, 1.0, 0.0])
[eager] computed Field(I[1:2]: [inf])
AssertionError: non-finite values in smooth
